In [1]:
# %% Qwen 학습용 JSONL 생성 (텔롭 텍스트만)
import os
import json
import re
import polars as pl
from tqdm import tqdm

# -------------------------
# Paths / Config
# -------------------------
OCR_DIR = "./data/3_ocr_results"
TELOP_DIR = "./data/4_is_telop_results"
STT_DIR = "./data/4_stt_results"
OUTPUT_DIR = "./data"

SYSTEM_PROMPT = """당신은 유튜브 영상의 텔롭 텍스트를 생성하는 모델입니다.

입력: 채널명, 영상명, 프레임별 STT(음성 인식) 시퀀스
출력: 각 프레임에 표시될 텔롭 텍스트 (여러 개면 | 구분)

텔롭: 자막, 캡션, 이름표, 채널 로고, 타임스탬프, 리액션 텍스트 등 후편집으로 삽입된 화면 텍스트
STT는 음성이고 텔롭은 화면 텍스트이므로 동일하지 않습니다."""

# -------------------------
# 직렬화
# -------------------------
def strip_video_id(video_name: str) -> str:
    return re.sub(r"_[A-Za-z0-9_-]{11}$", "", video_name)


def serialize_stt_input(channel_name: str, video_name: str, frames: list[dict]) -> str:
    lines = [
        f"채널: {channel_name}",
        f"영상: {strip_video_id(video_name)}",
        "",
    ]
    for f in frames:
        fnum = f["frame_num"]
        stt = f["stt_text"] if f["stt_text"] is not None else "null"
        lines.append(f"[F{fnum:04d}] {stt}")
    return "\n".join(lines)


def serialize_telop_output(frames: list[dict]) -> str:
    lines = []
    for f in frames:
        fnum = f["frame_num"]
        ocr_texts = json.loads(f["ocr_texts"]) if isinstance(f["ocr_texts"], str) else f["ocr_texts"]
        is_telop = json.loads(f["is_telop"]) if isinstance(f["is_telop"], str) else f["is_telop"]

        # telop만 필터링
        telop_texts = [t for t, flag in zip(ocr_texts, is_telop) if flag is True]

        if telop_texts:
            text = " | ".join(telop_texts)
        else:
            text = "null"
        lines.append(f"[F{fnum:04d}] {text}")
    return "\n".join(lines)


def build_sample(channel_name: str, video_name: str, ocr_path: str, telop_path: str, stt_path: str) -> dict | None:
    try:
        ocr_df = pl.read_parquet(ocr_path, glob=False)
        telop_df = pl.read_parquet(telop_path, glob=False)
        stt_df = pl.read_parquet(stt_path, glob=False)
    except Exception as e:
        print(f"⚠️ 파일 읽기 실패: {e}")
        return None

    # frame_num 기준 join: ocr + telop + stt
    merged = (
        ocr_df
        .join(telop_df, on="frame_num", how="left")
        .join(stt_df, on="frame_num", how="left")
        .sort("frame_num")
    )

    frames = merged.to_dicts()
    if not frames:
        return None

    user_text = serialize_stt_input(channel_name, video_name, frames)
    assistant_text = serialize_telop_output(frames)

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ],
        "metadata": {
            "channel": channel_name,
            "video": video_name,
            "num_frames": len(frames),
        },
    }


# -------------------------
# Main
# -------------------------
def main():
    channels = sorted([
        d for d in os.listdir(OCR_DIR)
        if os.path.isdir(os.path.join(OCR_DIR, d))
    ])
    print(f"📁 총 {len(channels)}개 채널 발견")

    all_videos = []
    for channel_name in channels:
        ocr_channel_dir = os.path.join(OCR_DIR, channel_name)
        telop_channel_dir = os.path.join(TELOP_DIR, channel_name)
        stt_channel_dir = os.path.join(STT_DIR, channel_name)

        if not os.path.isdir(stt_channel_dir):
            continue
        if not os.path.isdir(telop_channel_dir):
            continue

        for pf in sorted(os.listdir(ocr_channel_dir)):
            if not pf.endswith(".parquet"):
                continue
            video_name = pf[:-8]
            ocr_path = os.path.join(ocr_channel_dir, pf)
            telop_path = os.path.join(telop_channel_dir, pf)
            stt_path = os.path.join(stt_channel_dir, pf)

            if not os.path.exists(stt_path):
                continue
            if not os.path.exists(telop_path):
                continue

            all_videos.append((channel_name, video_name, ocr_path, telop_path, stt_path))

    print(f"🎯 {len(all_videos):,}개 영상 대상")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_DIR, "5_qwen_dataset.jsonl")
    f_out = open(out_path, "w", encoding="utf-8")

    n_saved = 0
    n_skipped = 0
    token_counts = []

    for channel_name, video_name, ocr_path, telop_path, stt_path in tqdm(all_videos, desc="샘플 생성"):
        sample = build_sample(channel_name, video_name, ocr_path, telop_path, stt_path)
        if sample is None:
            n_skipped += 1
            continue

        f_out.write(json.dumps(sample, ensure_ascii=False) + "\n")
        n_saved += 1

        total_chars = sum(len(m["content"]) for m in sample["messages"])
        token_counts.append(int(total_chars * 1.5))

    f_out.close()

    print(f"\n✅ 완료!")
    print(f"  저장: {n_saved:,}개")
    print(f"  스킵: {n_skipped:,}개")

    if token_counts:
        avg = sum(token_counts) / len(token_counts)
        mx = max(token_counts)
        print(f"  추정 토큰: 평균 {avg:,.0f}, 최대 {mx:,.0f}")

if __name__ == "__main__":
    main()

📁 총 664개 채널 발견
🎯 66,400개 영상 대상


샘플 생성: 100%|██████████| 66400/66400 [08:55<00:00, 124.10it/s]


✅ 완료!
  저장: 66,400개
  스킵: 0개
  추정 토큰: 평균 45,736, 최대 838,165
